# Tesis PUCP · Colab de verificación experimental
## PINNs `tanh` para Poisson 2D con Dirichlet fuerte

**Objeto.** Investigar numéricamente las condiciones de aproximación y regularidad utilizadas en la extensión condicional del Teorema 2 del Capítulo III. Se distinguen la optimización (A), el efecto de la anchura a muestreo fijo (B) y el escalamiento conjunto (C).

**Naturaleza de la evidencia.** Este cuaderno **reproduce un protocolo computacional y sus diagnósticos discretos**, pero **no demuestra** interpolación exacta, existencia de minimizadores globales, cotas Hölder continuas ni una tasa asintótica para todas las redes. Los resultados finitos son exploratorios.

**Software:** Python, PyTorch (autodiferenciación), NumPy, pandas, matplotlib. **Hardware recomendado:** GPU A100, precisión `float64`. Se guarda el entorno y un manifiesto SHA-256 para auditoría.

**Referencia teórica principal:** Shin, Darbon y Karniadakis (2020), *On the convergence of physics informed neural networks for linear second-order elliptic and parabolic type PDEs*, [arXiv:2004.01806](https://arxiv.org/abs/2004.01806). La extensión con comparadores aproximados es la formulación condicional del Capítulo III de la tesis.

### Preguntas de investigación

1. **A — Optimización:** ¿continúa disminuyendo la pérdida residual al prolongar L-BFGS desde los checkpoints refinados de las anchuras 32 y 64? Para ejecutar A se requieren los cuatro checkpoints previos; sin ellos se registra la omisión sin sustituirlos artificialmente.
2. **B — Anchura:** manteniendo `N=40`, es decir, **1 521 puntos i.i.d. interiores**, ¿cómo varían la pérdida residual, el error de prueba y la regularidad para anchuras 32, 64 y 128?
3. **C — Escalamiento:** al crecer conjuntamente la anchura y el tamaño de muestra, ¿es la evolución observada compatible con una pérdida empírica de orden \(a_m^{\mathrm{int}}\)?

**Medición crucial:** el número \(N\) es un *índice de tamaño*, no una cuadrícula de entrenamiento. Los \(m_r=(N-1)^2\) puntos de colocación se muestrean uniformemente e independientemente en \((-1,1)^2\). La cuadrícula regular se utiliza **solo para el diagnóstico de \(H^1\)**.

### Ecuación, arquitectura y cantidades que se comparan

Se utiliza una solución fabricada de Poisson:

\[
U=(-1,1)^2,\quad \Delta u=f,\quad u|_{\partial U}=0,
\qquad u^*(x,y)=\sin(\pi x)\sin(\pi y),\quad f=-2\pi^2u^*.
\]

La red impone la condición de Dirichlet **exactamente**:

\[
v_\theta(x,y)=(1-x^2)(1-y^2)N_\theta(x,y),
\qquad N_\theta:\; 2\to W\to W\to1,
\quad\sigma=\tanh.
\]

La pérdida observada en entrenamiento es la media cuadrática residual
\[
\widehat\varepsilon_m(v_\theta)=\frac1{m_r}\sum_{i=1}^{m_r}|\Delta v_\theta(X_i)-f(X_i)|^2.
\]

Para el cuadrado y las constantes geométricas adoptadas en la tesis, con \(\alpha=1/2\), \(c_r=1/9\), \(C_r=\pi/4\), \(\lambda_r=1\), se utiliza:

\[
\varepsilon_r=3\sqrt2\,m_r^{-1/4},
\qquad a_m^{\mathrm{int}}=\frac{2\sqrt2}{3\pi}\,m_r^{-3/4}.
\]

La fórmula se utiliza dentro del **régimen geométrico \(\varepsilon_r\le1\)**. Comparamos \(\widehat\varepsilon_m/a_m^{\mathrm{int}}\) y estimamos, usando pares finitos de puntos,
\(\max_{(x,y)\in P}|r(x)-r(y)|/\|x-y\|^{1/2}\), con \(r=\Delta v_\theta-f\). También se estima la misma cantidad para \(\Delta v_\theta\). **No se confunden estas estimaciones discretas con las seminormas continuas del teorema.**

### Protocolo de datos, selección y trazabilidad

- Dos semillas independientes (7 y 17); idénticos puntos de entrenamiento para distintas anchuras dentro de una misma semilla y un mismo \(N\). En C las muestras son anidadas por construcción, porque se reutiliza la semilla del generador al aumentar \(m_r\).
- El entrenamiento usa minibatches Adam y después etapas L-BFGS sobre un **subconjunto fijo**. La pérdida empírica presentada en las tablas, sin embargo, se **reevalúa sobre los \(m_r\) puntos completos**. Por ello no se finge una minimización exacta de toda la pérdida empírica.
- Se selecciona el mejor checkpoint mediante **validación**, jamás mediante prueba. La prueba se evalúa **una vez al final** para cada combinación.
- La evaluación de \(H^1\) proporciona dos números distintos: `H1_RMS_malla` (media cuadrática normalizada, comparable a reportes anteriores) y `H1_integral_aprox` (cuadratura trapezoidal de la **norma integral** \(H^1(U)\)); ninguno es certificado riguroso.
- Todos los checkpoints, historiales, CSV y resultados llevan respaldo a Drive cuando está disponible. Las repeticiones con la misma configuración se pueden reanudar; un run terminado no se vuelve a entrenar.

### 1. Configuración reproducible

En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → A100 GPU**. Ejecutar las celdas en orden. El perfil predeterminado `completo` ejecuta B y C; A se ejecuta cuando se encuentran sus checkpoints originales. Para comprobar rápidamente el código, poner `PERFIL = 'rapido'`. El perfil `extendido` añade \(N=150, W=256\) a C.

La sesión puede cerrarse: los CSV y checkpoints se respaldan en `Mi unidad/Tesis_PUCP/Colab_Jurado_Convergencia_Tanh`. Cada nueva ejecución restaura los archivos ya completados.

In [ ]:
# -*- coding: utf-8 -*-
"""Protocolos B y C: PINN tanh, Poisson 2D, Dirichlet fuerte.
Código de respaldo para el cuaderno autocontenido de sustentación.
Se conserva deliberadamente la misma arquitectura y a_m del experimento anterior.
"""
import os, sys, json, math, time, random, hashlib, shutil, platform, warnings, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

# ------------------------------------------------------
# CONFIGURACION, EDITABLE POR EL JURADO
# ------------------------------------------------------
PERFIL = os.environ.get('PINN_JURADO_PERFIL', 'completo')  # 'completo' | 'rapido' | 'extendido'
EJECUTAR_B = True
EJECUTAR_C = True
EJECUTAR_A_DESDE_CHECKPOINTS = True  # si faltan checkpoints, se omite, sin inventar datos
REUTILIZAR_COINCIDENCIAS_B_C = True
A_CONFIGS = [(32,40),(64,75)]
A_ETAPAS = 6
A_LBFGS_PUNTOS = 768
EVALUAR_HISTORICO_AB = True
MONTAR_DRIVE = True
SEMILLAS = [7, 17]
B_ANCHURAS = [32, 64, 128]
B_N = 40
C_CONFIG = [(40,64), (75,128), (110,192)]
ADAM_EPOCAS = 900     # pasos de minibatch, no épocas completas
ADAM_BLOQUE = 150
ADAM_BATCH = 128
ADAM_LR = 1e-3
LBFGS_ETAPAS = 4
LBFGS_MAX_ITER = 30
LBFGS_PUNTOS = 512
VALIDACION_M = 1024
PRUEBA_M = 2048
HOLDER_PARES = 1200
HOLDER_PARES_DENSO = 4800  # segundo diagnostico independiente de sensibilidad
LOTE_EVALUACION = 192
ALPHA = 0.5
DTYPE = torch.float64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if PERFIL == 'rapido':
    SEMILLAS = [7]
    B_ANCHURAS = [16, 32]
    B_N = 24
    EJECUTAR_A_DESDE_CHECKPOINTS=False
    C_CONFIG = [(24,16),(40,32)]
    ADAM_EPOCAS=12; ADAM_BLOQUE=6; LBFGS_ETAPAS=2; LBFGS_MAX_ITER=2
    LBFGS_PUNTOS=48; VALIDACION_M=64; PRUEBA_M=96; HOLDER_PARES=80; HOLDER_PARES_DENSO=120
    LOTE_EVALUACION=48; MONTAR_DRIVE=False
elif PERFIL == 'extendido':
    C_CONFIG.append((150,256))
elif PERFIL != 'completo':
    raise ValueError('PERFIL debe ser completo, rapido o extendido')

if DEVICE.type == 'cpu':
    torch.set_num_threads(min(os.cpu_count() or 2, 4))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# No todos los kernels son deterministas en todas las combinaciones GPU/PyTorch.
torch.use_deterministic_algorithms(True, warn_only=True)

ROOT = Path(os.environ.get('PINN_JURADO_ROOT','/content/Tesis_PUCP_Jurado_PINN' if os.path.exists('/content') else '/mnt/data/colab_jurado/salida'))
ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = None
DRIVE_HIST = None
if MONTAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_OUT = Path('/content/drive/MyDrive/Tesis_PUCP/Colab_Jurado_Convergencia_Tanh')
        DRIVE_HIST = Path('/content/drive/MyDrive/Tesis_PUCP/experimento_tanh_AB')
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    except ImportError:
        print('Fuera de Colab; se guardara solo localmente.')
    except Exception as exc:
        warnings.warn('Drive no disponible: '+str(exc))

# Recuperacion de ejecuciones anteriores, sin alterar originales.
if DRIVE_OUT:
    for src in DRIVE_OUT.rglob('*'):
        if src.is_file():
            dst=ROOT/src.relative_to(DRIVE_OUT)
            if not dst.exists():
                dst.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(src,dst)


def copiar_drive(p):
    p=Path(p)
    if DRIVE_OUT and p.is_file():
        target=DRIVE_OUT / p.relative_to(ROOT)
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,target)


def guardar_json(payload,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(payload,indent=2,ensure_ascii=False,default=str),encoding='utf8')
    os.replace(tmp,path);copiar_drive(path)


def guardar_csv(df,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.csv.tmp')
    df.to_csv(tmp,index=False,float_format='%.17g')
    os.replace(tmp,path);copiar_drive(path)


def guardar_modelo(payload,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.pt.tmp')
    torch.save(payload,tmp);os.replace(tmp,path);copiar_drive(path)


def state_cpu(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}


def reproducible(s):
    random.seed(s);np.random.seed(s);torch.manual_seed(s)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(s)


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

print('Perfil:',PERFIL,' | dispositivo:',DEVICE,' | precisión:',DTYPE)
print('PyTorch:',torch.__version__,' | CUDA:',torch.version.cuda,
      '| GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')
print('Directorio de salida:',ROOT,' | Respaldo Drive:',DRIVE_OUT)

### 2. Definiciones y controles matemáticos de la implementación

La prueba unitaria verifica el signo del término fuente, las segundas derivadas de la solución fabricada, el Dirichlet fuerte y el valor de la escala teórica para 1 521 puntos.

In [ ]:
# ------------------------------------------------------
# DEFINICION MATEMATICA Y ESCALA TEORICA
# ------------------------------------------------------
class RedTanhDirichlet(nn.Module):
    def __init__(self,ancho):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(2,ancho),nn.Tanh(),
                               nn.Linear(ancho,ancho),nn.Tanh(),nn.Linear(ancho,1))
    def forward(self,x):
        return (1-x[:,:1]**2)*(1-x[:,1:2]**2)*self.net(x)


def solucion(x):
    return torch.sin(math.pi*x[:,:1])*torch.sin(math.pi*x[:,1:2])


def fuente(x):
    return -2*math.pi**2 * solucion(x)


def epsilon_cobertura(m):
    # epsilon_r=sqrt(2)*(1/9)^(-1/2)*m^(-1/4)=3 sqrt(2) m^(-1/4)
    return 3*math.sqrt(2)*m**(-0.25)


def peso_interior(m):
    # lambda=1, alpha=1/2, C_r=pi/4, c_r=1/9.
    # 3 epsilon_r^(2alpha) / [3(C_r/c_r)*2 sqrt(m)]
    return 2*math.sqrt(2)/(3*math.pi)*m**(-0.75)


def rand_dominio(m,semilla):
    rng=torch.Generator(device='cpu').manual_seed(int(semilla))
    return 2*torch.rand(int(m),2,generator=rng,dtype=DTYPE)-1


def datos(N,semilla):
    m=(N-1)**2
    return {
       'train':rand_dominio(m,10000+semilla),
       'val':rand_dominio(VALIDACION_M,40000+semilla),
       'test':rand_dominio(PRUEBA_M,50000+semilla),
    }


def residuo(modelo,xy,optimizar=False):
    with torch.enable_grad():
        x=xy.detach().requires_grad_(True)
        u=modelo(x)
        gu=torch.autograd.grad(u.sum(),x,create_graph=True)[0]
        uxx=torch.autograd.grad(gu[:,0].sum(),x,create_graph=optimizar,retain_graph=True)[0][:,:1]
        uyy=torch.autograd.grad(gu[:,1].sum(),x,create_graph=optimizar)[0][:,1:2]
        return uxx+uyy-fuente(x)


def mse_residual(modelo,xy):
    valores=[]
    modelo.eval()
    for part in xy.split(LOTE_EVALUACION):
        r=residuo(modelo,part.to(DEVICE),False)
        valores.append(r.detach().square().cpu())
    return float(torch.cat(valores).mean())


def pares_regularidad(semilla, n_pairs=None):
    n_pairs = HOLDER_PARES if n_pairs is None else int(n_pairs)
    gen=torch.Generator(device='cpu').manual_seed(300000+semilla)
    x=2*torch.rand(n_pairs,2,generator=gen,dtype=DTYPE)-1
    theta=2*math.pi*torch.rand(n_pairs,1,generator=gen,dtype=DTYPE)
    rho=torch.exp(math.log(0.005)+torch.rand(n_pairs,1,generator=gen,dtype=DTYPE)*math.log(1.5/.005))
    y=(x+rho*torch.cat((theta.cos(),theta.sin()),dim=1)).clamp(-1,1)
    den=torch.norm(x-y,dim=1,keepdim=True).clamp_min(1e-12).pow(ALPHA)
    return x,y,den


def eval_residuo_puntos(modelo,x):
    return torch.cat([residuo(modelo,p.to(DEVICE),False).detach().cpu() for p in x.split(LOTE_EVALUACION)])


def holder_discreto(modelo,pares):
    x,y,den=pares
    rx=eval_residuo_puntos(modelo,x);ry=eval_residuo_puntos(modelo,y)
    # En la ecuacion de Poisson, Delta(v)=r+f.
    fx=fuente(x);fy=fuente(y)
    return {'holder_residuo_discreto':float(((rx-ry).abs()/den).max()),
            'holder_laplaciano_discreto':float(((rx+fx-ry-fy).abs()/den).max())}


def h1_grid(model,n=41):
    """Devuelve RMS H1 normalizada y aproximacion integral H1 mediante trapecios 2D."""
    axis=torch.linspace(-1,1,n,dtype=DTYPE)
    xx,yy=torch.meshgrid(axis,axis,indexing='ij')
    grid=torch.stack([xx.flatten(),yy.flatten()],dim=1)
    err=[]
    for part in grid.split(LOTE_EVALUACION):
        with torch.enable_grad():
            x=part.to(DEVICE).detach().requires_grad_(True)
            u=model(x)
            du=torch.autograd.grad(u.sum(),x,create_graph=False)[0]
            v=solucion(x)
            ux=math.pi*torch.cos(math.pi*x[:,:1])*torch.sin(math.pi*x[:,1:2])
            uy=math.pi*torch.sin(math.pi*x[:,:1])*torch.cos(math.pi*x[:,1:2])
            square=(u-v).square()+(du[:,:1]-ux).square()+(du[:,1:2]-uy).square()
            err.append(square.detach().cpu())
    square=torch.cat(err).reshape(n,n)
    w=torch.ones(n,dtype=DTYPE);w[0]=w[-1]=0.5
    quadrature=(square*w[:,None]*w[None,:]).sum()*(2/(n-1))**2
    return {'H1_RMS_malla':math.sqrt(float(square.mean())),
            'H1_integral_aprox':math.sqrt(float(quadrature))}


def controles_matematicos():
    # Pruebas unitarias para evitar errores de signo en f, gradientes y contorno.
    pts=rand_dominio(80,1234).to(DEVICE)
    with torch.enable_grad():
        x=pts.detach().requires_grad_(True)
        u=solucion(x)
        du=torch.autograd.grad(u.sum(),x,create_graph=True)[0]
        uxx=torch.autograd.grad(du[:,0].sum(),x,create_graph=True)[0][:,:1]
        uyy=torch.autograd.grad(du[:,1].sum(),x,create_graph=True)[0][:,1:2]
        max_r=float((uxx+uyy-fuente(x)).abs().max().detach().cpu())
    red=RedTanhDirichlet(8).to(DEVICE,dtype=DTYPE)
    t=torch.linspace(-1,1,23,dtype=DTYPE,device=DEVICE)
    borde=torch.cat((torch.stack([t,torch.ones_like(t)],1),
                     torch.stack([t,-torch.ones_like(t)],1),
                     torch.stack([torch.ones_like(t),t],1),
                     torch.stack([-torch.ones_like(t),t],1)),0)
    max_b=float(red(borde).abs().max().detach().cpu())
    assert max_r<1e-10,(max_r,'inconsistencia de signo o derivadas')
    assert max_b<1e-12,(max_b,'Dirichlet no es exacto')
    assert abs(peso_interior(1521)-0.00123219)<1e-8
    assert epsilon_cobertura(1521)<1.0
    return {'max_error_residuo_solucion_exacta':max_r,'max_error_dirichlet_red':max_b,
            'a_m_m1521':peso_interior(1521),'epsilon_r_m1521':epsilon_cobertura(1521)}

CHECKS=controles_matematicos()
print('Controles matemáticos:',CHECKS)

### 3. Entrenamiento reproducible y evaluación de validación/prueba

Se guarda una trayectoria completa para cada combinación. L-BFGS usa un subconjunto fijo de entrenamiento, elegido sin mirar los datos de prueba. El mejor checkpoint se selecciona exclusivamente por pérdida de validación.

In [ ]:
# ------------------------------------------------------
# METODOS DE OPTIMIZACION CON VALIDACION Y CHECKPOINTS
# ------------------------------------------------------
def evaluacion(modelo,d,pares,m_full=False):
    resp={'eps_train':mse_residual(modelo,d['train']),
          'eps_val':mse_residual(modelo,d['val'])}
    resp.update(holder_discreto(modelo,pares))
    if m_full:
        resp['eps_test']=mse_residual(modelo,d['test'])
        resp.update(h1_grid(modelo))
    return resp


def train_one(protocolo,N,W,seed):
    assert protocolo in {'B','C'}
    path=ROOT/protocolo/f'W{W}_N{N}_s{seed}'
    path.mkdir(parents=True,exist_ok=True)
    final=path/'resultado.json'
    if final.exists() and (path/'modelo_final.pt').exists():
        print(f'{protocolo} W{W} N{N} s{seed}: reutilizando resultado respaldado.')
        return json.loads(final.read_text(encoding='utf8'))

    reproducible(900000+seed)
    model=RedTanhDirichlet(W).to(DEVICE,dtype=DTYPE)
    d=datos(N,seed)
    h=pares_regularidad(seed)
    m=len(d['train']);a=peso_interior(m)
    base=evaluacion(model,d,h,m_full=False)
    best_val=base['eps_val'];best_state=state_cpu(model)
    hist=[]
    historypath=path/'historial.csv'
    progress=path/'progreso.pt'
    adam=torch.optim.Adam(model.parameters(),lr=ADAM_LR)
    rng=torch.Generator(device='cpu').manual_seed(80000+seed)
    adam_count=0;lbfgs_stage=0
    if progress.exists():
        snap=torch.load(progress,map_location='cpu',weights_only=True)
        assert snap['N']==N and snap['W']==W and snap['seed']==seed
        model.load_state_dict(snap['model'])
        best_state=snap['best'];best_val=float(snap['best_val'])
        adam_count=int(snap['adam_count']);lbfgs_stage=int(snap['lbfgs_stage'])
        rng.set_state(snap['batch_rng'])
        adam.load_state_dict(snap['adam_state'])
        for st in adam.state.values():
            for k,v in list(st.items()):
                if torch.is_tensor(v):st[k]=v.to(DEVICE)
        if historypath.exists(): hist=pd.read_csv(historypath).to_dict('records')
        print(f'{protocolo} W{W} N{N} semilla {seed}: reanuda Adam={adam_count}, LBFGS={lbfgs_stage}.')
    else:
        hist.append(dict(fase='inicio',pasos_adam=0,etapa_lbfgs=0,**base))
        guardar_csv(pd.DataFrame(hist),historypath)
        print(f'{protocolo} W{W} N{N} semilla {seed}: inicio nuevo, a_m={a:.6g}')

    def checkpoint():
        guardar_modelo({'N':N,'W':W,'seed':seed,'model':state_cpu(model),
                        'best':best_state,'best_val':best_val,
                        'adam_count':adam_count,'lbfgs_stage':lbfgs_stage,
                        'batch_rng':rng.get_state(),'adam_state':adam.state_dict()}, progress)

    while adam_count<ADAM_EPOCAS:
        nsteps=min(ADAM_BLOQUE,ADAM_EPOCAS-adam_count)
        model.train()
        for _ in range(nsteps):
            ids=torch.randint(m,(min(ADAM_BATCH,m),),generator=rng)
            adam.zero_grad(set_to_none=True)
            loss=residuo(model,d['train'][ids].to(DEVICE),True).square().mean()
            loss.backward();adam.step()
        adam_count+=nsteps
        evalr=evaluacion(model,d,h)
        if evalr['eps_val']<best_val:
            best_val=evalr['eps_val'];best_state=state_cpu(model)
        hist.append(dict(fase='Adam',pasos_adam=adam_count,etapa_lbfgs=0,**evalr))
        guardar_csv(pd.DataFrame(hist),historypath);checkpoint()
        print(f'  Adam {adam_count:4d}: train/a={evalr["eps_train"]/a:.3g}; valid={evalr["eps_val"]:.4g}')

    g=torch.Generator(device='cpu').manual_seed(81000+seed)
    chosen=torch.randperm(m,generator=g)[:min(m,LBFGS_PUNTOS)]
    xfix=d['train'][chosen].to(DEVICE)
    for stage in range(lbfgs_stage+1,LBFGS_ETAPAS+1):
        optimizer=torch.optim.LBFGS(model.parameters(),lr=0.7,max_iter=LBFGS_MAX_ITER,
                                    history_size=30,line_search_fn='strong_wolfe')
        calls=[0]
        def closure():
            calls[0]+=1
            optimizer.zero_grad(set_to_none=True)
            objective=residuo(model,xfix,True).square().mean()
            objective.backward()
            return objective
        model.train();optimizer.step(closure)
        lbfgs_stage=stage
        ev=evaluacion(model,d,h)
        if ev['eps_val']<best_val:
            best_val=ev['eps_val'];best_state=state_cpu(model)
        hist.append(dict(fase='LBFGS',pasos_adam=adam_count,etapa_lbfgs=stage,
                         evaluaciones_closure=calls[0],**ev))
        guardar_csv(pd.DataFrame(hist),historypath);checkpoint()
        print(f'  L-BFGS {stage}: train/a={ev["eps_train"]/a:.3g}; valid={ev["eps_val"]:.4g}')
        del optimizer

    model.load_state_dict(best_state)
    end=evaluacion(model,d,h,m_full=True) # SOLO AHORA se evalua el test independiente
    end['holder_residuo_denso']=holder_discreto(model,pares_regularidad(seed,HOLDER_PARES_DENSO))['holder_residuo_discreto']
    guardar_modelo(dict(state_dict=state_cpu(model),W=W,N=N,seed=seed,protocolo=protocolo),
                   path/'modelo_final.pt')
    result=dict(protocolo=protocolo,ancho=W,N=N,semilla=seed,m_r=m,a_m=a,
                epsilon_r=epsilon_cobertura(m),radios_en_regimen=epsilon_cobertura(m)<=1,
                eps_inicial=base['eps_train'],eps_final=end['eps_train'],
                eps_test=end['eps_test'],eps_val=end['eps_val'],
                razon_eps_a=end['eps_train']/a,
                razon_eps_test_a=end['eps_test']/a,
                holder_residuo=end['holder_residuo_discreto'],
                holder_residuo_denso=end['holder_residuo_denso'],
                holder_laplaciano=end['holder_laplaciano_discreto'],
                H1_RMS_malla=end['H1_RMS_malla'],H1_integral_aprox=end['H1_integral_aprox'],
                modelo=str(path/'modelo_final.pt'))
    guardar_json(result,final)
    del model,adam
    if torch.cuda.is_available():torch.cuda.empty_cache()
    gc.collect()
    return result

### 4. Continuación opcional del experimento histórico A

La prueba A **no puede reproducirse desde cero sin los pesos originales**. Este bloque busca los checkpoints `tanh_refinado_W{32|64}_N{40|75}_s{7|17}.pt` en las carpetas usadas previamente. Si no existen, se informa expresamente de la omisión. La prueba B y el escalamiento C son autocontenidos.

In [ ]:
# ------------------------------------------------------
# PROTOCOLO A: CONTINUACION OPCIONAL DE CHECKPOINTS HISTORICOS
# ------------------------------------------------------
def localizar_checkpoint_refinado(W,N,seed):
    name=f'tanh_refinado_W{W}_N{N}_s{seed}.pt'
    sources=[Path('/content/estudio_tanh_C2alpha/refinamiento_residuo')/name,
             Path('/content/drive/MyDrive/Tesis_PUCP/experimento_tanh_AB/fuentes_refinadas')/name,
             Path('/content/drive/MyDrive/Tesis_PUCP/refinamiento_residuo')/name]
    return next((p for p in sources if p.exists()),None)


def reentrenamiento_A(W,N,seed):
    fuente_cp=localizar_checkpoint_refinado(W,N,seed)
    if fuente_cp is None:
        print(f'A W{W} N{N} semilla{seed}: sin checkpoint refinado original. No se sustituye por una red nueva.')
        return None
    folder=ROOT/'A'/f'W{W}_N{N}_s{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    existing=folder/'resultado.json'
    if existing.exists() and (folder/'modelo_final.pt').exists():
        return json.loads(existing.read_text(encoding='utf8'))
    reproducible(900000+seed)
    model=RedTanhDirichlet(W).to(DEVICE,dtype=DTYPE)
    cp=torch.load(fuente_cp,map_location='cpu',weights_only=True)
    model.load_state_dict(cp['state_dict'])
    d=datos(N,seed);holder=pares_regularidad(seed)
    m=len(d['train']);a=peso_interior(m)
    base=evaluacion(model,d,holder)
    best_val=base['eps_val'];best_state=state_cpu(model)
    stages_done=0
    prog=folder/'progreso.pt'
    histfile=folder/'historial.csv'
    hist=[dict(etapa=0,**base)]
    if prog.exists():
        p=torch.load(prog,map_location='cpu',weights_only=True)
        model.load_state_dict(p['ultimo']);best_state=p['mejor']
        best_val=float(p['best_val']);stages_done=int(p['etapa'])
        if histfile.exists():hist=pd.read_csv(histfile).to_dict('records')
    else:guardar_csv(pd.DataFrame(hist),histfile)
    rng=torch.Generator(device='cpu').manual_seed(61000+seed)
    index=torch.randperm(m,generator=rng)[:min(A_LBFGS_PUNTOS,m)]
    fixed=d['train'][index].to(DEVICE)
    print(f'A W{W} N{N} seed={seed}: desde {fuente_cp.name}; etapas {stages_done}/{A_ETAPAS}')
    for etapa in range(stages_done+1,A_ETAPAS+1):
        opt=torch.optim.LBFGS(model.parameters(),lr=.7,max_iter=LBFGS_MAX_ITER,
                              history_size=30,line_search_fn='strong_wolfe')
        def closure():
            opt.zero_grad(set_to_none=True)
            loss=residuo(model,fixed,True).square().mean()
            loss.backward()
            return loss
        model.train();opt.step(closure)
        ev=evaluacion(model,d,holder)  # No observa el test hasta el final.
        if ev['eps_val']<best_val:
            best_val=ev['eps_val'];best_state=state_cpu(model)
        hist.append(dict(etapa=etapa,**ev))
        guardar_csv(pd.DataFrame(hist),histfile)
        guardar_modelo(dict(ultimo=state_cpu(model),mejor=best_state,
                            etapa=etapa,best_val=best_val),prog)
        print(f'  L-BFGS etapa {etapa}: eps/a={ev["eps_train"]/a:.4g}; val={ev["eps_val"]:.4g}')
        del opt
    model.load_state_dict(best_state)
    end=evaluacion(model,d,holder,m_full=True)
    end['holder_residuo_denso']=holder_discreto(model,pares_regularidad(seed,HOLDER_PARES_DENSO))['holder_residuo_discreto']
    res=dict(protocolo='A',ancho=W,N=N,semilla=seed,m_r=m,a_m=a,
             epsilon_r=epsilon_cobertura(m),radios_en_regimen=epsilon_cobertura(m)<=1,
             eps_inicial=base['eps_train'],eps_final=end['eps_train'],
             eps_test=end['eps_test'],eps_val=end['eps_val'],
             razon_eps_a=end['eps_train']/a,razon_eps_test_a=end['eps_test']/a,
             holder_residuo=end['holder_residuo_discreto'],
             holder_residuo_denso=end['holder_residuo_denso'],
             holder_laplaciano=end['holder_laplaciano_discreto'],
             H1_RMS_malla=end['H1_RMS_malla'],H1_integral_aprox=end['H1_integral_aprox'],
             origen_checkpoint=str(fuente_cp),sha256_fuente=sha256_file(fuente_cp),
             modelo=str(folder/'modelo_final.pt'))
    guardar_modelo(dict(state_dict=state_cpu(model),W=W,N=N,seed=seed,protocolo='A'),
                   folder/'modelo_final.pt')
    guardar_json(res,existing)
    del model
    if torch.cuda.is_available():torch.cuda.empty_cache()
    gc.collect()
    return res

### 5. Auditoría opcional del CSV histórico A+B

Cuando esté disponible `MyDrive/Tesis_PUCP/experimento_tanh_AB/resultados_A_B.csv`, se importan los datos originales, se verifica su escala aritmética y se registra su SHA-256. **Importar un CSV no equivale a reproducir el entrenamiento.** La gráfica histórica separa los valores finales para evitar que las pérdidas iniciales enormes oculten las tendencias.

In [ ]:
# ------------------------------------------------------
# IMPORTACION DE EVIDENCIA HISTORICA (SIN RENOMBRARLA COMO REPRODUCCION)
# ------------------------------------------------------
def auditar_historico():
    if not EVALUAR_HISTORICO_AB:return None
    candidates=[]
    if DRIVE_HIST: candidates.append(DRIVE_HIST/'resultados_A_B.csv')
    candidates += [Path('/content/estudio_tanh_C2alpha/experimento_AB/resultados_A_B.csv'),
                   Path('/content/drive/MyDrive/Tesis_PUCP/experimento_tanh_AB/resultados_A_B.csv')]
    src=next((x for x in candidates if x.exists()),None)
    if src is None:
        print('A+B anteriores: CSV no encontrado. No se inventan resultados historicos.')
        return None
    df=pd.read_csv(src)
    cols=['experimento','ancho','N','semilla','a_m_interior','eps_sobre_a_despues',
          'despues_holder_residuo','despues_eps_test']
    if any(c not in df.columns for c in cols):
        print('CSV historico con esquema no reconocido:',[c for c in cols if c not in df.columns]);return None
    errors=[]
    for _,r in df.iterrows():
        m=(int(r['N'])-1)**2
        if not np.isclose(float(r['a_m_interior']),peso_interior(m),rtol=1e-7):
            errors.append(f'peso teorico: experimento {r["experimento"]}, W{r["ancho"]}')
        if not np.isfinite(float(r['despues_eps_test'])) or float(r['despues_eps_test'])<0:
            errors.append('perdida test invalida')
    print('Archivo historico:',src,'SHA256:',sha256_file(src))
    if errors:print('ADVERTENCIAS en CSV historico:',errors)
    else:print('Verificacion aritmetica del CSV historico: OK (NO implica que los entrenamientos hayan sido reproducidos)')
    out=ROOT/'importacion_historica.csv';guardar_csv(df,out)
    return df

### 6. Tablas y figuras científicamente interpretables

Las gráficas B y C separan tres variables: error residual, cociente con la escala \(a_m\) y regularidad discreta. Se guardan las dos semillas y además las medianas y el rango mínimo–máximo. Con solo dos semillas **no** se atribuye significación estadística ni se reportan intervalos de confianza ficticios.

In [ ]:
# ------------------------------------------------------
# TABLAS Y FIGURAS DE SUSTENTACION
# ------------------------------------------------------
def resumir_y_graficar(df):
    if df is None or df.empty:return None
    grp=df.groupby(['protocolo','ancho','N','m_r'],as_index=False)
    res=grp.agg(n_semillas=('semilla','count'),
                mediana_eps=('eps_final','median'),
                minimo_eps=('eps_final','min'),maximo_eps=('eps_final','max'),
                mediana_ratio=('razon_eps_a','median'),
                mediana_eps_test=('eps_test','median'),
                mediana_holder=('holder_residuo','median'),
                mediana_holder_denso=('holder_residuo_denso','median'),
                mediana_H1_integral=('H1_integral_aprox','median'))
    guardar_csv(res,ROOT/'resumen_mediana_y_rango.csv')
    print('\nMEDIANAS Y RANGOS. No son intervalos de confianza.')
    print(res.to_string(index=False,float_format=lambda x:f'{x:.6g}'))
    for protocolo in ['B','C']:
        data=res[res.protocolo==protocolo].sort_values('ancho' if protocolo=='B' else 'm_r')
        if len(data)==0:continue
        x=data['ancho'].to_numpy() if protocolo=='B' else data['m_r'].to_numpy()
        fig,axes=plt.subplots(1,3,figsize=(16,4),constrained_layout=True)
        axes[0].semilogy(x,data['mediana_eps'],'o-',label='MSE entrenamiento')
        axes[0].semilogy(x,data['mediana_eps_test'],'s-',label='MSE prueba independiente')
        if protocolo=='C':
            axes[0].semilogy(x,[peso_interior(m) for m in x],'--',label='a_m (referencia)')
        axes[0].legend()
        axes[1].plot(x,data['mediana_ratio'],'o-')
        axes[1].set_ylabel('MSE entrenamiento / a_m')
        axes[2].plot(x,data['mediana_holder'],'o-',label='Seminorma residual discreta')
        axes[2].legend()
        for ax in axes:
            ax.set_xlabel('Anchura W' if protocolo=='B' else 'Puntos interiores m_r')
            ax.grid(alpha=.25)
        axes[0].set_title('Error y escala teorica')
        axes[1].set_title('Diagnostico de comparadores')
        axes[2].set_title('Regularidad residual estimada')
        path=ROOT/f'figura_{protocolo}_jurado.png'
        fig.savefig(path,dpi=190,bbox_inches='tight')
        copiar_drive(path)
        plt.show();plt.close(fig)
    # pendiente en C: estimador descriptivo, no teorema ni tasa certificada.
    cc=res[res.protocolo=='C'].sort_values('m_r')
    if len(cc)>=3:
        m=cc['m_r'].to_numpy(dtype=float);eps=cc['mediana_eps'].to_numpy(dtype=float)
        slope=np.polyfit(np.log(m),np.log(eps),1)[0]
        print(f'\nEXPLORATORIO: pendiente log(MSE) vs log(m_r) = {slope:.3f}; criterio de referencia -0.750.')
        print('Hay solo',len(cc),'escalas y',int(cc['n_semillas'].min()),'semillas por escala; NO es una verificacion asintotica.')
    return res


def figura_historica_AB(df):
    if df is None:return
    med=df.groupby(['experimento','ancho'],as_index=False).median(numeric_only=True)
    fig,ax=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
    for exp in ('A','B'):
        t=med[med['experimento']==exp]
        if len(t)==0:continue
        ax[0].plot(t.ancho,t.eps_sobre_a_despues,'o-',label=f'{exp} final')
        ax[1].plot(t.ancho,t.despues_holder_residuo,'o-',label=f'{exp} final')
    ax[0].set(title='A+B historicos: solo RESULTADOS FINALES',xlabel='Anchura',ylabel='MSE / a_m')
    ax[1].set(title='Seminorma Hölder residual discreta',xlabel='Anchura',ylabel='Maximo sobre pares evaluados')
    for a in ax:a.grid(alpha=.25);a.legend()
    path=ROOT/'historico_AB_finales_sin_escala_enganosa.png'
    fig.savefig(path,dpi=190,bbox_inches='tight');copiar_drive(path)
    plt.show();plt.close(fig)


def manifest_files():
    files=[]
    for p in sorted(ROOT.rglob('*')):
        if p.is_file() and p.name!='manifest_SHA256.csv':
            files.append({'ruta':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    csv=ROOT/'manifest_SHA256.csv';guardar_csv(pd.DataFrame(files),csv)
    return csv

### 7. Ejecución del protocolo y persistencia

El perfil `completo` contempla:

| Estudio | \(N\) | \(m_r\) | Anchuras | Semillas |
|---|---:|---:|---|---|
| A (condicional a checkpoints) | 40, 75 | 1 521; 5 476 | 32; 64 | 7, 17 |
| B (desde cero) | 40 | 1 521 | 32, 64, 128 | 7, 17 |
| C (desde cero) | 40, 75, 110 | 1 521; 5 476; 11 881 | 64, 128, 192 | 7, 17 |

La configuración común \((N,W)=(40,64)\) puede reutilizar **exactamente el mismo entrenamiento** en B y C: se identifica expresamente como dato compartido y **no** se cuenta como un ensayo independiente adicional. Para C se añade opcionalmente \((N,W)=(150,256)\) con el perfil `extendido`.

In [ ]:
# ------------------------------------------------------
# EJECUCION PRINCIPAL
# ------------------------------------------------------
metadata=dict(fecha_UTC=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
              python=sys.version,plataforma=platform.platform(),torch=torch.__version__,
              numpy=np.__version__,pandas=pd.__version__,
              GPU=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
              perfil=PERFIL,semillas=SEMILLAS,B_anchuras=B_ANCHURAS,B_N=B_N,
              C_config=C_CONFIG,A_configs=A_CONFIGS,A_desde_checkpoints=EJECUTAR_A_DESDE_CHECKPOINTS,ADAM_pasos=ADAM_EPOCAS,LBFGS_etapas=LBFGS_ETAPAS,
              LBFGS_max_iter=LBFGS_MAX_ITER,LBFGS_subset=LBFGS_PUNTOS,
              holder_pares=HOLDER_PARES,holder_pares_denso=HOLDER_PARES_DENSO,validacion_m=VALIDACION_M,prueba_m=PRUEBA_M,
              criterios='seleccion del mejor por validacion; test usado solo al final',
              controles_matematicos=CHECKS)
guardar_json(metadata,ROOT/'entorno_y_protocolo.json')
historico=auditar_historico()
if historico is not None:figura_historica_AB(historico)

rows=[]
if EJECUTAR_A_DESDE_CHECKPOINTS:
    print('\n==== PROTOCOLO A: CONTINUACION DE CHECKPOINTS (opcional) ====')
    for NandW in A_CONFIGS:
        W,N=NandW
        for seed in SEMILLAS:
            rr=reentrenamiento_A(W,N,seed)
            if rr is not None:
                rows.append(rr)
                guardar_csv(pd.DataFrame(rows),ROOT/'resultados_reproducidos_parciales.csv')
if EJECUTAR_B:
    print('\n==== PROTOCOLO B: ANCHURA VARIABLE, MUESTREO FIJO ====')
    for W in B_ANCHURAS:
        for seed in SEMILLAS:
            rows.append(train_one('B',B_N,W,seed))
            guardar_csv(pd.DataFrame(rows),ROOT/'resultados_reproducidos_parciales.csv')
if EJECUTAR_C:
    print('\n==== PROTOCOLO C: ESCALAMIENTO CONJUNTO (EXPLORATORIO) ====')
    for N,W in C_CONFIG:
        for seed in SEMILLAS:
            coincide=next((r for r in rows if r['protocolo']=='B' and
                           r['N']==N and r['ancho']==W and r['semilla']==seed),None)
            if REUTILIZAR_COINCIDENCIAS_B_C and coincide is not None:
                rr=dict(coincide)
                rr['protocolo']='C'
                rr['proveniencia']='reutiliza exactamente la misma realizacion del protocolo B'
                print(f'C W{W} N{N} s{seed}: reutiliza B identico; NO se cuenta como ensayo independiente.')
                rows.append(rr)
            else:
                rows.append(train_one('C',N,W,seed))
            guardar_csv(pd.DataFrame(rows),ROOT/'resultados_reproducidos_parciales.csv')
if rows:
    final=pd.DataFrame(rows)
    guardar_csv(final,ROOT/'resultados_reproducidos.csv')
    resumir_y_graficar(final)
manifest_files()
print('Resultados reproducidos:',ROOT)
print('Respaldo:',DRIVE_OUT if DRIVE_OUT else 'solo local')
print('LIMITACION: el experimento NO prueba hipotesis de interpolacion, supremum Holder continuo,')
print('            minimizacion global, ni una ley O(a_m) para todos los m_r.')

### 8. Qué puede concluir un jurado — y qué no

**Las cifras que se generen al ejecutar este cuaderno sustentan afirmaciones finitas:** evolución de la pérdida residual bajo el protocolo de optimización fijado, comparación de arquitecturas a muestreo fijo, errores en puntos de prueba independientes, error \(H^1\) aproximado y regularidad residual sobre pares de puntos.

**No sustentan una demostración de** \(\varepsilon_m=O(a_m)\) para todo \(m\), ni una cota de seminormas Hölder continuas, ni existencia de comparadores de pérdida suficientemente pequeña para **toda** la secuencia, ni una garantía de minimización global o control del *gap* \(\delta_m\). Incluso una pendiente empírica favorable en tres tamaños muestrales constituye **evidencia exploratoria, no prueba asintótica**.

**La distinción entre capacidades:** B mantiene fijo el número de puntos y varía la anchura; C varía ambas cantidades, por lo que una mejora en C no se puede atribuir exclusivamente a la anchura. A solo evalúa el efecto de continuar la optimización desde checkpoints concretos. 

**Procedencia de las gráficas históricas:** se importan de los CSV reales si existen. El cuaderno no contiene resultados numéricos simulados ni curvas generadas a partir de datos ficticios.

### 9. Exportación opcional para depósito ante el jurado

Esta celda crea un ZIP con configuración, métricas, historiales, figuras, checkpoints y manifiesto de integridad. No se ejecuta automáticamente para evitar duplicar archivos voluminosos.

In [ ]:
import shutil
from pathlib import Path
archivo_entrega = shutil.make_archive(str(ROOT.parent / "Tesis_PUCP_Colab_Jurado_Resultados"), "zip", root_dir=ROOT)
print("Paquete de resultados:", archivo_entrega)
print("El cuaderno .ipynb debe adjuntarse por separado como fuente reproducible.")
# Para descargar desde Colab, ejecutar opcionalmente:
# from google.colab import files
# files.download(archivo_entrega)